# CUSTOMER CHURN & REVENUE INTELLIGENCE
## Part 1: Exploratory Data Analysis (EDA) & Monetization KPIs
**Author:** Senior Data Analyst & BI Developer  
**Dataset:** Telecom / SaaS Customer Base (8,500 records)  
**Business Goal:** Identify customer attrition drivers, quantify revenue losses, and connect customer behavioral metrics directly to corporate financial health.

---
### Table of Contents:
1. Executive Problem Framing & Environment Setup
2. Data Loading & Schema Verification
3. Core Business & Monetization KPIs
4. Customer Churn & Retention Distributions
5. Revenue Contribution by Subscription Plan
6. Contract Type & Payment Method Interaction
7. Customer Lifecycle (Tenure Cohort) Analysis
8. Service Friction & Support Ticket Escalations
9. Usage Hours & Engagement Levels
10. Root Cause Attribution: Churn Reasons & Financial Losses
11. Statistical Feature Correlations
12. Strategic Analytical Summary


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Visual display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Segoe UI', 'DejaVu Sans', 'Arial'

print("Analytics environment initialized successfully.")


## 1. Data Ingestion & Verification
We load the verified clean customer dataset containing 8,500 records across 18 behavioral and financial dimensions.

In [ ]:
df = pd.read_csv('../dataset/customer_churn_cleaned.csv')
print(f"Dataset Shape: {df.shape[0]:,} rows by {df.shape[1]} columns")
df.head(5)


## 2. Core Business Financial & Retention KPIs
In SaaS and subscription business models, customer attrition must be measured in both **unit customer volume** and **Monthly Recurring Revenue (MRR)**.

In [ ]:
total_customers = len(df)
active_customers = (df['Churn_Status'] == 'No').sum()
churned_customers = (df['Churn_Status'] == 'Yes').sum()
churn_rate = churned_customers / total_customers * 100
retention_rate = active_customers / total_customers * 100

active_mrr = df[df['Churn_Status'] == 'No']['Monthly_Charges'].sum()
mrr_lost = df[df['Churn_Status'] == 'Yes']['Monthly_Charges'].sum()
total_potential_mrr = df['Monthly_Charges'].sum()
arpu_active = df[df['Churn_Status'] == 'No']['Monthly_Charges'].mean()
arpu_blended = df['Monthly_Charges'].mean()
total_historical_rev = df['Total_Charges'].sum()
avg_clv = df['Total_Charges'].mean()

kpi_summary = pd.DataFrame({
    'Metric Category': [
        'Customer Volume', 'Customer Volume', 'Customer Volume', 'Attrition Metric', 'Attrition Metric',
        'Financial Run-Rate', 'Financial Exposure', 'Financial Base', 'Unit Economics', 'Unit Economics',
        'Cumulative Monetization', 'Customer Value'
    ],
    'KPI Name': [
        'Total Customer Base', 'Active Retained Customers', 'Churned Customers', 'Overall Churn Rate', 'Retention Rate',
        'Active Monthly Recurring Revenue (MRR)', 'Monthly Revenue Lost to Churn', 'Total Potential Monthly Billing',
        'ARPU (Active Base)', 'ARPU (Blended Portfolio)', 'Cumulative Historical Revenue', 'Average Customer Lifetime Value (CLV)'
    ],
    'Value': [
        f"{total_customers:,}", f"{active_customers:,}", f"{churned_customers:,}", f"{churn_rate:.2f}%", f"{retention_rate:.2f}%",
        f"${active_mrr:,.2f}", f"${mrr_lost:,.2f}", f"${total_potential_mrr:,.2f}", f"${arpu_active:.2f}", f"${arpu_blended:.2f}",
        f"${total_historical_rev:,.2f}", f"${avg_clv:,.2f}"
    ],
    'Business Interpretation': [
        'Total tracked subscribers in database', 'Currently paying and active subscriber base', 'Subscribers who terminated subscriptions',
        '31.74% annual portfolio attrition', '68.26% customer retention benchmark',
        'Contracted recurring cash flow per month ($5.65M ARR)', 'Recurring revenue bleeding each month ($2.47M ARR loss)',
        'Total addressable revenue without churn', 'Average monthly spend per active paying user', 'Blended monthly realization across all users',
        'Total lifetime cash collected from inception', 'Mean lifetime cash contribution per account'
    ]
})

kpi_summary


### Key Financial Finding:
The company experiences an overall customer churn rate of **31.74%**, which translates to an ongoing monthly revenue destruction of **$205,776.91** (**$2,469,322.92 annually**). The current active Monthly Recurring Revenue (MRR) stands at **$470,499.45** ($5.65M ARR).

## 3. Subscription Plan Monetization & Attrition Breakdown
Let's analyze how Monthly Recurring Revenue and customer churn distribute across subscription tiers (Basic, Standard, Premium, Enterprise).

In [ ]:
plan_analysis = df.groupby('Subscription_Plan').agg(
    Total_Customers=('CustomerID', 'count'),
    Active_Customers=('Churn_Status', lambda x: (x == 'No').sum()),
    Churned_Customers=('Churn_Status', lambda x: (x == 'Yes').sum()),
    Active_MRR=('Monthly_Charges', lambda x: x[df.loc[x.index, 'Churn_Status'] == 'No'].sum()),
    Lost_MRR=('Monthly_Charges', lambda x: x[df.loc[x.index, 'Churn_Status'] == 'Yes'].sum()),
    Avg_Monthly_Charge=('Monthly_Charges', 'mean'),
    Avg_CLV=('Total_Charges', 'mean')
).reset_index()

plan_analysis['Churn_Rate_%'] = (plan_analysis['Churned_Customers'] / plan_analysis['Total_Customers'] * 100).round(2)
plan_analysis['%_of_Total_MRR'] = (plan_analysis['Active_MRR'] / active_mrr * 100).round(2)
plan_analysis.sort_values(by='Active_MRR', ascending=False)


## 4. Contract Type & Payment Method Interaction
Contract structure represents the single largest determinant of customer retention. Month-to-Month contracts provide customer flexibility but represent acute flight risk.

In [ ]:
contract_matrix = df.groupby(['Contract_Type', 'Payment_Method']).agg(
    Total=('CustomerID', 'count'),
    Churned=('Churn_Status', lambda s: (s == 'Yes').sum()),
    MRR_Lost=('Monthly_Charges', lambda s: s[df.loc[s.index, 'Churn_Status'] == 'Yes'].sum())
).reset_index()

contract_matrix['Churn_Rate_%'] = (contract_matrix['Churned'] / contract_matrix['Total'] * 100).round(2)
contract_matrix.sort_values(by='Churn_Rate_%', ascending=False)


### Critical Finding:
Customers on **Month-to-Month contracts paying via Electronic Check** suffer an alarming **84.85% churn rate**, accounting for **$87,348.18 in monthly lost revenue** alone! Switching customers to automated payment methods (Direct Bank Transfer or Credit Card) cuts churn to ~51-53% on monthly plans, and below 4% on multi-year plans.

## 5. Tenure Lifecycle: The Onboarding Cliff
Customer attrition is heavily front-loaded in the subscription lifecycle.

In [ ]:
tenure_summary = df.groupby('Tenure_Cohort', observed=False).agg(
    Total_Customers=('CustomerID', 'count'),
    Churned_Customers=('Churn_Status', lambda s: (s == 'Yes').sum()),
    MRR_Lost=('Monthly_Charges', lambda s: s[df.loc[s.index, 'Churn_Status'] == 'Yes'].sum()),
    Avg_Usage=('Usage_Hours', 'mean')
).reset_index()

tenure_summary['Churn_Rate_%'] = (tenure_summary['Churned_Customers'] / tenure_summary['Total_Customers'] * 100).round(2)
tenure_summary['%_of_All_Churns'] = (tenure_summary['Churned_Customers'] / churned_customers * 100).round(2)
tenure_summary


## 6. Support Friction & Customer Dissatisfaction
Customer support interactions serve as an early warning signal of churn.

In [ ]:
support_perf = df.groupby('Support_Tickets').agg(
    Customer_Count=('CustomerID', 'count'),
    Active_Count=('Churn_Status', lambda s: (s == 'No').sum()),
    Churned_Count=('Churn_Status', lambda s: (s == 'Yes').sum()),
    Avg_Monthly_Charges=('Monthly_Charges', 'mean')
).reset_index()

support_perf['Churn_Rate_%'] = (support_perf['Churned_Count'] / support_perf['Customer_Count'] * 100).round(2)
support_perf


## 7. Churn Reason Attribution & Financial Losses
Quantifying stated churn reasons by monthly and annualized revenue loss.

In [ ]:
churn_reasons_df = df[df['Churn_Status'] == 'Yes'].groupby('Churn_Reason').agg(
    Churned_Count=('CustomerID', 'count'),
    Monthly_Revenue_Lost=('Monthly_Charges', 'sum'),
    Avg_Monthly_Charge=('Monthly_Charges', 'mean'),
    Avg_Tenure=('Tenure_Months', 'mean')
).reset_index()

churn_reasons_df['Annualized_Loss'] = churn_reasons_df['Monthly_Revenue_Lost'] * 12
churn_reasons_df['%_of_Total_Lost_MRR'] = (churn_reasons_df['Monthly_Revenue_Lost'] / mrr_lost * 100).round(2)
churn_reasons_df.sort_values(by='Monthly_Revenue_Lost', ascending=False)


## 8. Summary of Analytical Findings
1. **Total Churn Exposure:** 2,698 churned customers represent **$205,776.91 in monthly lost revenue** ($2.47M ARR).
2. **Contract Vulnerability:** Month-to-month contracts account for 95%+ of all churn incidents.
3. **The 6-Month Onboarding Cliff:** Customers within their first 6 months exhibit a 71.37% churn rate, contributing to 47.22% of total company attrition.
4. **Payment Friction:** Electronic checks suffer an 84.85% churn rate on monthly contracts.
5. **Support Ticket Tipping Point:** When a customer logs 3 or more tickets, churn probability surges above 40%, reaching 80%+ at 5+ tickets.
6. **Competitor Pricing Pressure:** 31.28% of churned revenue is driven by competitors offering superior introductory pricing.
